# Computer Infrastructure – Assessment (Winter 25/26)

**Author:** Eder Olimpio

---

## Introduction

This notebook contains my solutions to the problems defined in `problems.md` for
the Computer Infrastructure module (Winter 25/26). The target audience is an
informed computing professional, so I focus on clear explanations, reproducible
code, and concise documentation rather than just minimal working solutions.

Each problem is presented with:

- A short description of the task and context
- A brief plan or approach
- The corresponding code cells
- A short discussion of the results

Where external libraries or tools are used (for example `yfinance`, `pandas`,
`matplotlib`, and GitHub Actions), I include inline links to their official
documentation in the relevant Markdown cells and explain why they are
appropriate for this project.

## Imports

In this notebook I use:

- `pandas` for data manipulation and CSV I/O
  ([documentation](https://pandas.pydata.org/docs/))
- `yfinance` for downloading historical FAANG stock data from Yahoo! Finance
  ([project page](https://github.com/ranaroussi/yfinance))
- `matplotlib.pyplot` for plotting time series and saving figures to PNG
  ([gallery](https://matplotlib.org/stable/gallery/index.html))

The `datetime`, `pathlib.Path`, and `typing` modules are part of the Python
standard library and are used for timestamps, filesystem paths, and type hints.

In [4]:
from datetime import datetime
from pathlib import Path
from typing import List, Optional

import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

## Problem 1 – Downloading FAANG data

### Problem description

The goal of this problem is to download five days of hourly price data for the
FAANG stocks (META, AAPL, AMZN, NFLX, GOOG) using the `yfinance` Python
package and save the results to a CSV file. The data should be stored in a
`data/` directory in the repository root, and the file name should include a
timestamp in the format `YYYYMMDD-HHmmss.csv`.

The `yfinance` library provides a convenient `yf.download()` function for
retrieving historical market data from Yahoo! Finance
([documentation](https://github.com/ranaroussi/yfinance)). When multiple
tickers are requested, it returns a pandas DataFrame with a MultiIndex over
the columns (price field and ticker symbol), which will be useful in later
problems.

### Plan

- Define a `get_data()` function that:
  - Accepts an optional list of tickers (defaulting to the FAANG list).
  - Ensures that a `data/` directory exists in the current working directory.
  - Calls `yf.download()` with `period="5d"` and `interval="1h"` to retrieve
    hourly data for the past five days.
  - Builds a timestamped filename in the required format.
  - Saves the DataFrame to CSV in `data/` and returns the `Path` to the file.

In [10]:
def get_data(
    tickers: Optional[List[str]] = None,
    period: str = "5d",
    interval: str = "1h",
) -> Path:
    """
    Download hourly stock data for the previous five days for the given tickers
    using the yfinance package, and save the results to a timestamped CSV file
    inside the `data` folder in the repository root.

    Args:
        tickers: List of ticker symbols to download. If None, FAANG_TICKERS is used.
        period: Period of historical data to download (default "5d").
        interval: Data interval (default "1h" for hourly).

    Returns:
        Path object pointing to the saved CSV file.
    """
    # Use default FAANG tickers if none are provided
    if tickers is None:
        tickers = FAANG_TICKERS

    # Ensure the data directory exists in the current working directory
    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)

    # Join ticker symbols into a space-separated string for yfinance
    tickers_str = " ".join(tickers)

    # Download the data (previous 5 days, hourly)
    df = yf.download(
        tickers=tickers_str,
        period=period,
        interval=interval,
        auto_adjust=False,  # keep raw OHLC data
        threads=True,
    )

    # Build a timestamped filename: YYYYMMDD-HHmmss.csv
    timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
    file_path = data_dir / f"{timestamp_str}.csv"

    # Save the DataFrame to CSV
    df.to_csv(file_path)

    return file_path

In [12]:
# Call get_data() and inspect the result
csv_path = get_data()
print(f"Data saved to: {csv_path}")

# Load the CSV back in and show the first rows.
# Note: header=[0, 1] recreates the MultiIndex columns produced by yfinance.
loaded_df = pd.read_csv(csv_path, header=[0, 1], index_col=0)
loaded_df.head()

NameError: name 'FAANG_TICKERS' is not defined

### Discussion

The `get_data()` function downloads five days of hourly data for the FAANG
stocks (META, AAPL, AMZN, NFLX, GOOG) using `yf.download()` with
`period="5d"` and `interval="1h"`
([yfinance documentation](https://github.com/ranaroussi/yfinance)). The
resulting pandas DataFrame has a column **MultiIndex** where the first level
is the price field (`Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume`)
and the second level is the ticker symbol.

When the data is saved to CSV and read back in with:

```python
pd.read_csv(csv_path, header=[0, 1], index_col=0)
```

pandas reconstructs this MultiIndex automatically
([MultiIndex guide](https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html#multiindex-advanced-indexing)).
This structure will be useful in later problems when selecting, for example,
all `Close` prices at once.

Using a timestamp in the filename (for example `20251201-123239.csv`) avoids
overwriting previous files and provides a clear record of when the data was
retrieved. The `data/` directory is created on demand with
`Path("data").mkdir(parents=True, exist_ok=True)`, which means the code can be
run on any machine without manually creating directories and keeps the
repository itself free of large data files.

## Problem 2 – Plotting FAANG close prices

### Problem description

Write a function called `plot_data()` that:

- Opens the **latest** data file in the `data` folder.
- On **one plot**, plots the **Close** prices for each of the five stocks
  (META, AAPL, AMZN, NFLX, GOOG).
- Includes axis labels, a legend, and the **date** as the plot title.
- Saves the plot into a `plots` folder in the root of the repository using a
  filename in the format `YYYYMMDD-HHmmss.png`.
- Creates the `plots` folder if it does not already exist.

### Plan

1. Find the latest CSV file in the `data` directory.
2. Load the data into a pandas DataFrame, keeping the MultiIndex columns.
3. Extract the `Close` prices for each ticker.
4. Plot all five Close price series on a single set of axes using Matplotlib,
   with:
   - x-axis label: datetime
   - y-axis label: close price (USD)
   - legend showing the ticker symbols
   - title showing the most recent date in the dataset
5. Save the plot into a `plots` directory with a timestamped filename and
   return the path.

In [19]:
def get_latest_data_file(data_dir: Path = Path("data")) -> Path:
    """
    Return the path to the most recently modified CSV file in the given
    data directory.

    Args:
        data_dir: Directory containing CSV files produced by get_data().

    Returns:
        Path to the latest CSV file.

    Raises:
        FileNotFoundError: If the data directory does not exist or
            contains no CSV files.
    """
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    csv_files = list(data_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    # Pick the file with the most recent modification time
    latest_file = max(csv_files, key=lambda p: p.stat().st_mtime)
    return latest_file

In [20]:
latest = get_latest_data_file()
latest

PosixPath('data/20251201-125211.csv')

In [21]:
def plot_data() -> Path:
    """
    Open the latest data CSV in the `data` folder and plot the Close prices
    for each of the five FAANG stocks on a single figure.

    The plot includes:
        - x-axis: datetime
        - y-axis: close price (USD)
        - legend: ticker symbols
        - title: date (based on the most recent timestamp in the data)

    The figure is saved into a `plots` directory in the repository root using
    a timestamped filename in the format YYYYMMDD-HHmmss.png.

    Returns:
        Path to the saved PNG file.
    """
    # Find the most recent CSV file produced by get_data()
    data_file = get_latest_data_file()

    # Read the CSV with a MultiIndex for columns (field, ticker)
    df = pd.read_csv(data_file, header=[0, 1], index_col=0)
    df.index = pd.to_datetime(df.index)

    # Extract the Close prices for each ticker
    close_df = df["Close"]  # columns are the ticker symbols

    # Prepare the plots directory
    plots_dir = Path("plots")
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot each ticker's Close price
    for ticker in close_df.columns:
        ax.plot(close_df.index, close_df[ticker], label=ticker)

    ax.set_xlabel("Datetime")
    ax.set_ylabel("Close price (USD)")

    # Use the latest timestamp in the data as the date in the title
    latest_timestamp = close_df.index.max()
    date_str = latest_timestamp.strftime("%Y-%m-%d")
    ax.set_title(f"FAANG Close Prices – {date_str}")

    ax.legend()
    fig.autofmt_xdate()  # make x-axis labels more readable

    # Build a timestamped filename for the plot
    timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
    plot_path = plots_dir / f"{timestamp_str}.png"

    # Save and close the figure
    fig.savefig(plot_path, bbox_inches="tight")
    plt.close(fig)

    return plot_path

In [22]:
plot_path = plot_data()
print(f"Plot saved to: {plot_path}")

Plot saved to: plots/20251203-111100.png


### Discussion

The `plot_data()` function first locates the most recent CSV file in the
`data` directory using `get_latest_data_file()`. The file is read into a
pandas DataFrame with a MultiIndex on the columns, and the `Close` prices for
each FAANG ticker are extracted into a separate DataFrame.

All five Close price series are plotted on a single Matplotlib figure with
datetime on the x-axis and close price (USD) on the y-axis. A legend
identifies each ticker, and the plot title shows the most recent date in the
data. The figure is written to a `plots` directory using a timestamped
filename (e.g. `20251201-130215.png`), which avoids overwriting existing
plots and keeps the output organised.

## Problem 3 – Command-line script

### Problem description

I need to create a Python script called `faang.py` in the root of the
repository. The script should contain the functions from Problems 1 and 2 and
be executable from the terminal as `./faang.py`. When executed, the script
should:

1. Download the latest FAANG data into the `data` folder.
2. Create a plot of the Close prices and save it into the `plots` folder.

This requires adding a shebang line to the script and marking the file as
executable on the command line.

### Approach and implementation

To solve this problem I created a standalone script `faang.py` in the root of
the repository and copied the logic from Problems 1 and 2 into reusable
functions:

- `get_data()` – downloads five days of hourly FAANG data into the `data`
  folder.
- `get_latest_data_file()` – selects the most recent CSV file in `data/`.
- `plot_data()` – loads the latest CSV and plots the Close prices for all five
  tickers into the `plots` folder.

I added a `main()` function that:

1. Calls `get_data()` and prints the path to the CSV file.
2. Calls `plot_data()` and prints the path to the generated PNG file.

The script starts with a shebang line:

```bash
#!/usr/bin/env python3
```

This allows the operating system to locate the correct Python interpreter when
running the script from the terminal. I then marked the script as executable
using:

```bash
chmod +x faang.py
```

Finally, I can run the whole workflow from the project root with:

```bash
./faang.py
```
which downloads fresh data into data/ and creates a new plot in plots/
without needing to open the notebook.

## Problem 4 – Automation with GitHub Actions

### Problem description

I need to create a GitHub Actions workflow that runs my `faang.py` script every
Saturday morning. The workflow file should be called `faang.yml` and stored in
`.github/workflows/` in the root of the repository. In this section I will
also explain each line of the workflow.

### Workflow explanation

The workflow is defined in `.github/workflows/faang.yml`:

```yaml
name: Run FAANG script

on:
  # Run every Saturday at 08:00 UTC (Saturday morning)
  schedule:
    - cron: "0 8 * * 6"
  # Allow manual runs from the GitHub Actions tab
  workflow_dispatch:

jobs:
  run-faang:
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.9"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          if [ -f requirements.txt ]; then
            python -m pip install -r requirements.txt
          else
            python -m pip install yfinance pandas matplotlib
          fi

      - name: Run FAANG script
        run: python faang.py
```

Line-by-line explanation:

- `name: Run FAANG script`
  Sets a readable name for the workflow that appears in the GitHub Actions UI.

- `on:`
  Defines the events that trigger the workflow.

- `schedule:`
  Tells GitHub Actions to run the workflow on a cron schedule.

- `- cron: "0 8 * * 6"`
  Cron expression meaning at 08:00 UTC every Saturday (`6` is Saturday).
  This satisfies the requirement to run the script every Saturday morning.

- `workflow_dispatch:`
  Adds a manual trigger so I can start the workflow on demand from the
  Actions tab in GitHub.

- `jobs:`
  Top-level section that defines one or more jobs. Here I have a single job
  called `run-faang`.

- `run-faang:`
  The id of the job that runs my FAANG pipeline.

- `runs-on: ubuntu-latest`
  Tells GitHub Actions to run the job on the latest Ubuntu Linux runner.

- `steps:`
  A job is made up of a sequence of steps that the runner executes.

- `- name: Check out repository`
  Human-readable name for the first step.

- `uses: actions/checkout@v4`
  Uses the official `actions/checkout` action to clone the repository onto the
  runner so that the workflow can access `faang.py` and other files.

- `- name: Set up Python`
  Starts a new step to configure Python.

- `uses: actions/setup-python@v5`
  Uses the official `setup-python` action to install and configure Python on
  the runner.

- `with: python-version: "3.9"`
  Requests Python 3.9, which matches the version used locally in the project.

- `- name: Install dependencies`
  Step that installs the Python packages needed by `faang.py`.

- `run: |`
  Runs a multi-line shell script on the runner.

- `python -m pip install --upgrade pip`
  Upgrades `pip` to a recent version.

- `if [ -f requirements.txt ]; then ... fi`
  If a `requirements.txt` file exists, install all dependencies from it.
  Otherwise, install the specific packages needed (`yfinance`, `pandas`,
  `matplotlib`). This keeps the workflow flexible.

- `- name: Run FAANG script`
  Final step that actually runs my pipeline.

- `run: python faang.py`
  Executes the `faang.py` script using the configured Python interpreter.
  This will download the latest FAANG data into `data/` and generate a plot
  into `plots/`, in the same way as when I run the script locally.